# Hackathon Judge Fine-tuning with Unsloth + SFT

Fine-tunes a small Qwen3 model to judge hackathon projects using:
- **Dataset**: `twangodev/devpost-hacks-judgments` — pairwise judgments from Qwen3.5-27B across multiple DevPost hackathons
- **LoRA** for parameter-efficient fine-tuning
- **SFT** for supervised fine-tuning — trains the model to imitate the frontier model's full reasoning chain and verdict

**Runs on:** Kaggle T4 (16GB VRAM) or any GPU with 16GB+ VRAM.

## 1. Install Dependencies

In [1]:
%%capture
!pip install unsloth unsloth_zoo trl datasets scikit-learn

## 2. Load the DevPost Dataset

Each row is one pairwise judgment made by Qwen3.5-27B (the frontier model).
- `messages`: full conversation — system prompt + two project descriptions + frontier model's reasoning + verdict
- `verdict`: frontier model's choice (`A`, `B`, `tie`, or `invalid`)
- `hackathon`: source hackathon name (used for stratified splitting)
- `position`: whether projects were shown as `ab` or `ba` (position-swapped pairs for bias checking)
- `gt_a_result` / `gt_b_result`: human ground truth results for each project

Load all hackathons or filter to one:
```python
ds = load_dataset("twangodev/devpost-hacks-judgments")                     # all
ds = load_dataset("twangodev/devpost-hacks-judgments", "treehacks-2026")   # one hackathon
```

## Dataset Validation

Run these checks before training to catch any schema issues early.

In [ ]:
import re

print('=== Dataset Validation ===')
errors = []

# 1. Required columns present
required_cols = ['messages', 'judgment_id', 'pair_id', 'hackathon', 'position',
                 'project_a_id', 'project_b_id', 'verdict', 'gt_a_result', 'gt_b_result', 'model']
missing = [c for c in required_cols if c not in df.columns]
if missing:
    errors.append(f'Missing columns: {missing}')
else:
    print(f'✓ All required columns present')

# 2. Messages structure: each row must have [system, user, assistant]
def check_messages(msgs):
    if not isinstance(msgs, list) or len(msgs) != 3:
        return False
    return [m.get('role') for m in msgs] == ['system', 'user', 'assistant']

bad_msgs = df[~df['messages'].apply(check_messages)]
if len(bad_msgs) > 0:
    errors.append(f'{len(bad_msgs)} rows have malformed messages')
else:
    print(f'✓ All {len(df)} rows have valid [system, user, assistant] structure')

# 3. Verdict values
valid_verdicts = {'A', 'B', 'tie', 'invalid'}
bad_verdicts = df[~df['verdict'].isin(valid_verdicts)]
if len(bad_verdicts) > 0:
    errors.append(f'{len(bad_verdicts)} rows have unexpected verdict values: {bad_verdicts["verdict"].unique()}')
else:
    print(f'✓ All verdicts valid: {df["verdict"].value_counts().to_dict()}')

# 4. Position values
bad_pos = df[~df['position'].isin(['ab', 'ba'])]
if len(bad_pos) > 0:
    errors.append(f'{len(bad_pos)} rows have unexpected position values: {bad_pos["position"].unique()}')
else:
    print(f'✓ All positions valid: {df["position"].value_counts().to_dict()}')

# 5. Each pair_id should appear exactly twice (ab + ba)
pair_counts = df.groupby('pair_id').size()
bad_pairs = pair_counts[pair_counts != 2]
if len(bad_pairs) > 0:
    print(f'⚠ {len(bad_pairs)} pair_ids do not appear exactly twice: {bad_pairs.head().to_dict()}')
else:
    print(f'✓ All {df["pair_id"].nunique()} pair_ids appear exactly twice (ab + ba)')

# 6. Assistant message contains VERDICT line
def has_verdict(msgs):
    content = msgs[2]['content'] if len(msgs) > 2 else ''
    return bool(re.search(r'VERDICT:\s*(A|B|tie|invalid)', content, re.IGNORECASE))

no_verdict = df[~df['messages'].apply(has_verdict)]
if len(no_verdict) > 0:
    print(f'⚠ {len(no_verdict)} assistant messages have no parseable VERDICT line')
else:
    print(f'✓ All assistant messages contain a VERDICT line')

# 7. Trainable rows summary
trainable = df[df['verdict'].isin(['A', 'B'])]
print(f'
✓ Trainable rows (verdict A or B): {len(trainable)} / {len(df)}')
print(f'  Filtered out: {len(df) - len(trainable)} (tie or invalid)')

# Summary
print('
=== Validation Summary ===')
if errors:
    for e in errors:
        print(f'✗ {e}')
    raise ValueError('Dataset validation failed — fix errors above before training')
else:
    print('✓ All checks passed — dataset is ready for training')


## 3. Format Dataset for SFT

SFT trains the model to reproduce the frontier model's full reasoning chain and verdict.
We keep the complete `messages` (system + user + assistant).

We filter out `invalid` and `tie` verdicts, then split by `pair_id` so both
position-swapped versions of a pair always land in the same split.

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
import collections

def format_for_sft(row):
    return {
        "messages":    row["messages"],
        "answer":      row["verdict"],
        "pair_id":     row["pair_id"],
        "hackathon":   row["hackathon"],
        "position":    row["position"],
        "gt_a_result": row["gt_a_result"],
        "gt_b_result": row["gt_b_result"],
    }

# Filter to only clean A/B verdicts
formatted = [format_for_sft(row) for _, row in df.iterrows()
             if row["verdict"] in ("A", "B")]

print(f"Total rows after filtering invalid/tie: {len(formatted)}")

# Split by pair_id — keeps both position variants together
unique_pairs = df[df["verdict"].isin(["A", "B"])]["pair_id"].unique()
train_pairs, test_pairs = train_test_split(unique_pairs, test_size=0.2, random_state=42)
train_pairs, test_pairs = set(train_pairs), set(test_pairs)

train_data = [ex for ex in formatted if ex["pair_id"] in train_pairs]
test_data  = [ex for ex in formatted if ex["pair_id"] in test_pairs]

train_dataset = Dataset.from_list(train_data)
test_dataset  = Dataset.from_list(test_data)

print(f"Train: {len(train_dataset)} examples ({len(train_pairs)} unique pairs)")
print(f"Test:  {len(test_dataset)} examples ({len(test_pairs)} unique pairs)")

print(f"\nHackathon distribution in train:")
train_hacks = collections.Counter(ex["hackathon"] for ex in train_data)
for h, n in sorted(train_hacks.items()):
    print(f"  {h}: {n}")

print(f"\nMessage roles: {[m['role'] for m in train_dataset[0]['messages']]}")
print(f"Frontier verdict: {train_dataset[0]['answer']}")
print(f"\nAssistant response preview:")
print(train_dataset[0]['messages'][2]['content'][:300] + "...")

## 4. Load Model with Unsloth

In [ ]:
from unsloth import FastModel
import torch

model_name = "unsloth/Qwen3-4B"

# Qwen3-1.7B fits on a free Colab T4.
# Swap to unsloth/Qwen3-4B if you have more VRAM.
model, tokenizer = FastModel.from_pretrained(
    # model_name = "unsloth/Qwen3-0.6B",  # smallest Qwen3 for testing
    model_name=model_name,
    max_seq_length=8192,
    load_in_4bit=True,
    full_finetuning=False,
)

print(f"Model loaded on: {next(model.parameters()).device}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.41G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded on: cuda:0


## 5. Apply LoRA Adapters

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    use_gradient_checkpointing="unsloth",
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Trainable: 6,422,528 / 1,041,228,800 (0.62%)


## 6. Verdict Parser (for Evaluation)

We still need `parse_verdict` to evaluate the model's outputs at test time.
During SFT training this isn't used — the loss is cross-entropy on the assistant tokens only.

In [6]:
import re

def parse_verdict(response: str):
    """
    Extract VERDICT: A/B/TIE from response.
    Matches the format used in the dataset's system prompt.
    """
    if "</think>" in response:
        response = response.split("</think>")[-1]

    match = re.search(r"VERDICT:\s*([AB]|TIE)", response, re.IGNORECASE)
    if match:
        return match.group(1).upper()

    # Fallback
    if re.search(r"\bProject\s+A\b", response, re.IGNORECASE): return "A"
    if re.search(r"\bProject\s+B\b", response, re.IGNORECASE): return "B"
    return None

# Quick check
print(parse_verdict("<think>reasoning</think>\nVERDICT: A"))  # A
print(parse_verdict("VERDICT: B"))                             # B
print(parse_verdict("I cannot decide"))                        # None

A
B
None


## 7. Train with SFT

`SFTTrainer` applies the chat template and trains only on the assistant tokens (not the prompt).
The model learns to reproduce the frontier model's reasoning and verdict.

In [7]:
from trl import SFTTrainer, SFTConfig

def preprocess(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_dataset_tokenized = train_dataset.map(preprocess, num_proc = 1)
test_dataset_tokenized  = test_dataset.map(preprocess, num_proc = 1)

training_args = SFTConfig(
    output_dir="./hackathon_judge_sft",
    num_train_epochs=1,
    gradient_checkpointing=True,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,  # increase this too
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=5,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=8192,
    dataset_text_field="text",  # point to the pre-formatted column
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    # no formatting_func needed
)

print(f"Training on {len(train_dataset)} examples ({len(train_pairs)} unique pairs)")
print(f"Method: SFT — cross-entropy loss on assistant tokens only")
print(f"Epochs: {training_args.num_train_epochs}")
print("="*60)
trainer.train()

Map (num_proc=1):   0%|          | 0/156 [00:00<?, ? examples/s]

Map (num_proc=1):   0%|          | 0/37 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/156 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Training on 156 examples (80 unique pairs)
Method: SFT — cross-entropy loss on assistant tokens only
Epochs: 1


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 156 | Num Epochs = 1 | Total steps = 20
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 6,422,528 of 1,726,997,504 (0.37% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
5,2.324890
10,2.070213
15,1.955118
20,1.917417


Unsloth: Restored added_tokens_decoder metadata in ./hackathon_judge_sft/checkpoint-20/tokenizer_config.json.


TrainOutput(global_step=20, training_loss=2.066909646987915, metrics={'train_runtime': 1240.57, 'train_samples_per_second': 0.126, 'train_steps_per_second': 0.016, 'total_flos': 7041076983705600.0, 'train_loss': 2.066909646987915, 'epoch': 1.0})

## 8. Save LoRA Weights

In [8]:
# Get the underlying PEFT model and save it
trainer.model.save_pretrained("hackathon_judge_lora")
tokenizer.save_pretrained("hackathon_judge_lora")
print("Saved.")

Unsloth: Restored added_tokens_decoder metadata in hackathon_judge_lora/tokenizer_config.json.


Saved.


## 9. Evaluate on Held-out Test Set

Measures agreement with the frontier model's verdict on unseen pairs.

In [ ]:
# Reload base model fresh, then attach the saved adapter
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name=model_name,
    max_seq_length=8192,
    load_in_4bit=True,
)

from peft import PeftModel
model = PeftModel.from_pretrained(model, "hackathon_judge_lora")
model.eval()
print("Loaded.")

def run_inference(prompt_messages):
    text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=8192,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
        )
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response, parse_verdict(response)


print("Test Set Evaluation")
print("="*60)

frontier_correct = 0
results = []

for ex in test_dataset:
    prompt_messages = [m for m in ex["messages"] if m["role"] != "assistant"]
    response, predicted = run_inference(prompt_messages)
    frontier_match = predicted == ex["answer"]
    frontier_correct += int(frontier_match)

    think_text = ""
    if "<think>" in response and "</think>" in response:
        think_text = response.split("<think>")[1].split("</think>")[0].strip()

    results.append({
        "pair_id": ex["pair_id"],
        "position": ex["position"],
        "predicted": predicted,
        "frontier_verdict": ex["answer"],
        "gt_a": ex["gt_a_result"],
        "gt_b": ex["gt_b_result"],
        "frontier_match": frontier_match,
    })

    print(f"\npair {ex['pair_id'][:8]} | pos={ex['position']}")
    print(f"  Predicted: {predicted} | Frontier: {ex['answer']} | {'✓' if frontier_match else '✗'}")
    print(f"  GT A: {ex['gt_a_result']}")
    print(f"  GT B: {ex['gt_b_result']}")
    if think_text:
        print(f"  Reasoning: {think_text[:200]}...")

print("\n" + "="*60)
print(f"Agreement with frontier model: {frontier_correct}/{len(test_dataset)} = {frontier_correct/len(test_dataset)*100:.1f}%")

==((====))==  Unsloth 2026.4.8: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

unsloth/qwen3-1.7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Loaded.
Test Set Evaluation


Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



pair 0da99fe9 | pos=ba
  Predicted: A | Frontier: A | ✓
  GT A: Did Not Place
  GT B: Did Not Place
  Reasoning: Okay, let's start comparing these two projects. Project A is called "The Maddest Mockers" and Project B is "A-Meal". I need to evaluate each based on the criteria provided.

First, originality and cre...


...


pair d313aede | pos=ab
  Predicted: A | Frontier: A | ✓
  GT A: Did Not Place
  GT B: Did Not Place
  Reasoning: Okay, let's dive into comparing these two projects. I need to evaluate both A and B based on the criteria provided. First, I'll start by understanding what each project is about and how they were buil...


Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



pair d313aede | pos=ba
  Predicted: B | Frontier: B | ✓
  GT A: Did Not Place
  GT B: Did Not Place
  Reasoning: Okay, so I need to compare these two hackathon projects, A and B, to determine which one is stronger. Let's start by breaking down each project based on the criteria provided.

First, let's look at th...


Both `max_new_tokens` (=8192) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



pair 2d06b232 | pos=ab
  Predicted: A | Frontier: A | ✓
  GT A: Did Not Place
  GT B: Winner Most Dangerous Hack
  Reasoning: Okay, I need to compare these two hackathon projects: Bucky's Buzzer Beater (Project A) and Electric Chair (Project B). The user wants me to evaluate each one based on several criteria: originality, t...

pair 72197b6a | pos=ba
  Predicted: B | Frontier: B | ✓
  GT A: Did Not Place
  GT B: Winner MadHacks First Place
  Reasoning: Okay, let's see. I need to compare these two projects for a hackathon. Project A is BlueGuide, and Project B is 3Docs. I need to evaluate them based on originality, technical depth, execution quality,...

Agreement with frontier model: 27/37 = 73.0%


## 10. Position Bias Check

Each pair appears twice with projects in opposite order.
A consistent model should pick the same underlying project regardless of which slot it appears in.
Inconsistency = position bias (model favors whichever project appears first).

In [10]:
results_df = pd.DataFrame(results)

print("Position Bias Analysis")
print("="*60)

n_consistent = 0
n_pairs_checked = 0

for pair_id in test_pairs:
    pair_rows = results_df[results_df["pair_id"] == pair_id]
    if len(pair_rows) < 2:
        continue

    ab_row = pair_rows[pair_rows["position"] == "ab"]
    ba_row = pair_rows[pair_rows["position"] == "ba"]
    if ab_row.empty or ba_row.empty:
        continue

    ab_verdict = ab_row.iloc[0]["predicted"]
    ba_verdict = ba_row.iloc[0]["predicted"]

    # Consistent: same underlying project wins in both orderings
    # ab=A means project_a_id won; ba=B means project_a_id won (since they were swapped)
    consistent = (ab_verdict == "A" and ba_verdict == "B") or \
                 (ab_verdict == "B" and ba_verdict == "A")

    n_consistent += int(consistent)
    n_pairs_checked += 1

    print(f"\npair {pair_id[:8]}:")
    print(f"  GT A: {ab_row.iloc[0]['gt_a']}")
    print(f"  GT B: {ab_row.iloc[0]['gt_b']}")
    print(f"  ab verdict: {ab_verdict} | ba verdict: {ba_verdict}")
    print(f"  Consistent: {'✓' if consistent else '✗ position bias detected'}")

print(f"\nConsistency: {n_consistent}/{n_pairs_checked} pairs = {n_consistent/n_pairs_checked*100:.0f}%")

Position Bias Analysis

pair f0ac362f:
  GT A: Did Not Place
  GT B: Did Not Place
  ab verdict: A | ba verdict: B
  Consistent: ✓

pair 1f0b4dd6:
  GT A: Winner Best Community Hack
  GT B: Did Not Place
  ab verdict: A | ba verdict: A
  Consistent: ✗ position bias detected

pair 72197b6a:
  GT A: Winner MadHacks First Place
  GT B: Did Not Place
  ab verdict: A | ba verdict: B
  Consistent: ✓

pair 07d70483:
  GT A: Did Not Place
  GT B: Did Not Place
  ab verdict: A | ba verdict: A
  Consistent: ✗ position bias detected

pair 32805f9b:
  GT A: Did Not Place
  GT B: Did Not Place
  ab verdict: None | ba verdict: B
  Consistent: ✗ position bias detected

pair 1c9920c1:
  GT A: Did Not Place
  GT B: Did Not Place
  ab verdict: A | ba verdict: B
  Consistent: ✓

pair 7b60c404:
  GT A: Did Not Place
  GT B: Did Not Place
  ab verdict: A | ba verdict: B
  Consistent: ✓

pair b12c6d46:
  GT A: Did Not Place
  GT B: Did Not Place
  ab verdict: B | ba verdict: A
  Consistent: ✓

pair 220a969e

## 11. Next Steps

**More hackathons**: Load specific hackathons via `load_dataset("twangodev/devpost-hacks-judgments", "treehacks-2026")` and see if performance varies by hackathon domain.

**Scale training**: Increase `num_train_epochs` to 2-3 with a larger dataset.

**Larger model**: Swap `Qwen3-1.7B` for `Qwen3-4B` (needs ~10GB VRAM with 4-bit).

**Bradley-Terry ranking**: Convert all pairwise verdicts to a full ranking:

```python
# pip install choix
import choix, numpy as np

# comparisons: list of (winner_idx, loser_idx) tuples
params = choix.lsr_pairwise(n_items=num_projects, data=comparisons)
rankings = np.argsort(params)[::-1]
```

**Human alignment eval**: Compare BT rankings from each model against human ground truth (`gt_a_result`, `gt_b_result`) to measure which model best replicates human judgment.

**Per-hackathon eval**: Break down accuracy by `hackathon` field to see if the model generalizes across different hackathon styles.